In [ ]:
# ===============================
# Cell 1: Imports
# ===============================
import numpy as np
import pandas as pd
from scipy.stats import norm
from scipy.optimize import brentq

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


In [ ]:
# ===============================
# Cell 2: Global config
# ===============================
TRADING_DAYS_PER_YEAR = 252
STEPS_PER_DAY = 4
STEPS_PER_YEAR = TRADING_DAYS_PER_YEAR * STEPS_PER_DAY  # 1008
SIGMA = 2.51
CONTRACT_SIZE = 3000

def weeks_to_steps(weeks: float) -> int:
    return int(round(weeks * 5 * STEPS_PER_DAY))

T_2W = weeks_to_steps(2)  # 40
T_3W = weeks_to_steps(3)  # 60
assert T_2W == 40 and T_3W == 60


In [ ]:
# ===============================
# Cell 3: Input contract table
# IMPORTANT: verify against UI / raw data from you
# ===============================
contracts = pd.DataFrame([
    # symbol, kind, strike, expiry_steps, choice_steps, barrier, payout, bid, ask, max_buy, max_sell
    {"symbol":"AC",        "kind":"spot",       "strike":np.nan, "expiry_steps":T_3W, "choice_steps":np.nan, "barrier":np.nan, "payout":np.nan, "bid":49.975, "ask":50.025, "max_buy":200, "max_sell":200},

    {"symbol":"AC_50_P",   "kind":"put",        "strike":50.0,   "expiry_steps":T_3W, "choice_steps":np.nan, "barrier":np.nan, "payout":np.nan, "bid":12.00,  "ask":12.05,  "max_buy":50,  "max_sell":50},
    {"symbol":"AC_50_C",   "kind":"call",       "strike":50.0,   "expiry_steps":T_3W, "choice_steps":np.nan, "barrier":np.nan, "payout":np.nan, "bid":12.00,  "ask":12.05,  "max_buy":50,  "max_sell":50},
    {"symbol":"AC_35_P",   "kind":"put",        "strike":35.0,   "expiry_steps":T_3W, "choice_steps":np.nan, "barrier":np.nan, "payout":np.nan, "bid":4.33,   "ask":4.35,   "max_buy":50,  "max_sell":50},
    {"symbol":"AC_40_P",   "kind":"put",        "strike":40.0,   "expiry_steps":T_3W, "choice_steps":np.nan, "barrier":np.nan, "payout":np.nan, "bid":6.50,   "ask":6.55,   "max_buy":50,  "max_sell":50},
    {"symbol":"AC_45_P",   "kind":"put",        "strike":45.0,   "expiry_steps":T_3W, "choice_steps":np.nan, "barrier":np.nan, "payout":np.nan, "bid":9.05,   "ask":9.10,   "max_buy":50,  "max_sell":50},
    {"symbol":"AC_60_C",   "kind":"call",       "strike":60.0,   "expiry_steps":T_3W, "choice_steps":np.nan, "barrier":np.nan, "payout":np.nan, "bid":8.80,   "ask":8.85,   "max_buy":50,  "max_sell":50},

    {"symbol":"AC_50_P_2", "kind":"put",        "strike":50.0,   "expiry_steps":T_2W, "choice_steps":np.nan, "barrier":np.nan, "payout":np.nan, "bid":9.70,   "ask":9.75,   "max_buy":50,  "max_sell":50},
    {"symbol":"AC_50_C_2", "kind":"call",       "strike":50.0,   "expiry_steps":T_2W, "choice_steps":np.nan, "barrier":np.nan, "payout":np.nan, "bid":9.70,   "ask":9.75,   "max_buy":50,  "max_sell":50},

    {"symbol":"AC_50_CO",  "kind":"chooser",    "strike":50.0,   "expiry_steps":T_3W, "choice_steps":T_2W,   "barrier":np.nan, "payout":np.nan, "bid":22.20,  "ask":22.30,  "max_buy":50,  "max_sell":50},
    {"symbol":"AC_40_BP",  "kind":"binary_put", "strike":40.0,   "expiry_steps":T_3W, "choice_steps":np.nan, "barrier":np.nan, "payout":10.0,   "bid":5.00,   "ask":5.10,   "max_buy":50,  "max_sell":50},
    {"symbol":"AC_45_KO",  "kind":"ko_put",     "strike":45.0,   "expiry_steps":T_3W, "choice_steps":np.nan, "barrier":35.0,   "payout":np.nan, "bid":0.15,   "ask":0.175,  "max_buy":500, "max_sell":500},
])

# Basic sanity
required_cols = {"symbol","kind","strike","expiry_steps","choice_steps","barrier","payout","bid","ask","max_buy","max_sell"}
assert required_cols.issubset(set(contracts.columns))
assert (contracts["bid"] <= contracts["ask"]).all(), "Found crossed quotes."

# infer S0 from spot midpoint
spot = contracts.loc[contracts["kind"] == "spot"].iloc[0]
S0 = float((spot["bid"] + spot["ask"]) / 2.0)
print("S0 inferred from spot midpoint:", S0)

contracts

In [ ]:
# ===============================
# Cell 4: Black-Scholes utilities + assumption tests
# ===============================
def bs_price(S, K, T, sigma, option_type="call"):
    if T <= 0:
        if option_type == "call":
            return max(S - K, 0.0)
        elif option_type == "put":
            return max(K - S, 0.0)
        else:
            raise ValueError("option_type must be 'call' or 'put'")
    volT = sigma * np.sqrt(T)
    d1 = (np.log(S / K) + 0.5 * sigma * sigma * T) / volT
    d2 = d1 - volT
    if option_type == "call":
        return S * norm.cdf(d1) - K * norm.cdf(d2)
    else:
        return K * norm.cdf(-d2) - S * norm.cdf(-d1)

def binary_put_price(S, K, T, sigma, payout):
    if T <= 0:
        return payout * (1.0 if S < K else 0.0)
    volT = sigma * np.sqrt(T)
    d2 = (np.log(S / K) - 0.5 * sigma * sigma * T) / volT
    return payout * norm.cdf(-d2)

def implied_vol(price, S, K, T, option_type="call"):
    # robust bracket for very high vol environment
    f = lambda v: bs_price(S, K, T, v, option_type) - price
    return brentq(f, 1e-6, 10.0)

def mid(symbol):
    r = contracts.loc[contracts["symbol"] == symbol].iloc[0]
    return 0.5 * (r["bid"] + r["ask"])

# Put-call parity checks at K=50 for both maturities
pc_3w = mid("AC_50_C") - mid("AC_50_P")
pc_2w = mid("AC_50_C_2") - mid("AC_50_P_2")
print("Put-Call parity check K=50:")
print("3W: C_mid - P_mid =", pc_3w, " target S0-K =", S0 - 50.0)
print("2W: C_mid - P_mid =", pc_2w, " target S0-K =", S0 - 50.0)

# Implied vols under intended mapping (3W=15 trading days, 2W=10 trading days)
T3 = T_3W / STEPS_PER_YEAR
T2 = T_2W / STEPS_PER_YEAR
iv_3w = implied_vol(mid("AC_50_C"), S0, 50.0, T3, "call")
iv_2w = implied_vol(mid("AC_50_C_2"), S0, 50.0, T2, "call")
print("\nImplied vol check vs sigma=2.51:")
print("IV 3W ATM call:", iv_3w)
print("IV 2W ATM call:", iv_2w)

# Optional "wrong mapping" diagnostic: treat T+21 as 21 trading days
T3_wrong = 21 * STEPS_PER_DAY / STEPS_PER_YEAR
iv_3w_wrong = implied_vol(mid("AC_50_C"), S0, 50.0, T3_wrong, "call")
print("\nIf you wrongly map T+21 as 21 trading days, IV would be:", iv_3w_wrong)
print("This should NOT match 2.51 if our 3W=15-trading-days interpretation is correct.")

In [ ]:
# ===============================
# Cell 5: Path simulation + payoffs
# ===============================
def simulate_paths_chunk(n_paths, n_steps, s0, sigma, rng):
    dt = 1.0 / STEPS_PER_YEAR
    drift = -0.5 * sigma * sigma * dt
    vol = sigma * np.sqrt(dt)
    z = rng.standard_normal((n_paths, n_steps))
    logS = np.log(s0) + np.cumsum(drift + vol * z, axis=1)
    return np.exp(logS)

def payoff_vector(row, S, s0):
    kind = row["kind"]
    e = int(round(row["expiry_steps"])) - 1
    ST = S[:, e]

    if kind == "spot":
        return ST

    K = float(row["strike"])

    if kind == "call":
        return np.maximum(ST - K, 0.0)

    if kind == "put":
        return np.maximum(K - ST, 0.0)

    if kind == "binary_put":
        payout = float(row["payout"])
        return payout * (ST < K)

    if kind == "chooser":
        c = int(round(row["choice_steps"])) - 1
        Sc = S[:, c]
        # Rule from prompt: choose side that is ITM at choice time
        return np.where(
            Sc >= K,
            np.maximum(ST - K, 0.0),
            np.maximum(K - ST, 0.0)
        )

    if kind == "ko_put":
        barrier = float(row["barrier"])
        path_min = np.minimum(s0, np.min(S[:, :e+1], axis=1))
        knocked = path_min < barrier  # strict "below"
        return np.where(knocked, 0.0, np.maximum(K - ST, 0.0))

    raise ValueError(f"Unknown kind: {kind}")

In [ ]:
# ===============================
# Cell 6: Monte Carlo fair values with confidence intervals
# ===============================
def price_all_mc(df_contracts, s0, sigma, n_paths=2_000_000, chunk_size=200_000, seed=42, verbose=True):
    max_steps = int(df_contracts["expiry_steps"].max())
    rng = np.random.default_rng(seed)

    n_instr = len(df_contracts)
    sums = np.zeros(n_instr, dtype=np.float64)
    sums2 = np.zeros(n_instr, dtype=np.float64)

    done = 0
    while done < n_paths:
        m = min(chunk_size, n_paths - done)
        S = simulate_paths_chunk(m, max_steps, s0, sigma, rng)

        for i, (_, row) in enumerate(df_contracts.iterrows()):
            p = payoff_vector(row, S, s0)
            sums[i] += np.sum(p)
            sums2[i] += np.dot(p, p)

        done += m
        if verbose and (done % (5 * chunk_size) == 0 or done == n_paths):
            print(f"MC progress: {done:,} / {n_paths:,}")

    fair = sums / n_paths
    var = np.maximum(sums2 / n_paths - fair * fair, 0.0)
    se = np.sqrt(var / n_paths)
    ci_low = fair - 1.96 * se
    ci_high = fair + 1.96 * se

    out = df_contracts.copy()
    out["fair_mc"] = fair
    out["se_mc"] = se
    out["ci_low"] = ci_low
    out["ci_high"] = ci_high
    return out


priced = price_all_mc(
    contracts,
    s0=S0,
    sigma=SIGMA,
    n_paths=2_000_000,      # increase to 5M+ for final precision
    chunk_size=200_000,
    seed=123,
    verbose=True
)

priced[["symbol","kind","bid","ask","fair_mc","se_mc","ci_low","ci_high"]]

In [ ]:
# ===============================
# Cell 7: Analytical cross-checks where available
# ===============================
def analytic_fair(row, s0, sigma):
    kind = row["kind"]
    T = float(row["expiry_steps"]) / STEPS_PER_YEAR

    if kind == "spot":
        return s0
    if kind == "call":
        return bs_price(s0, float(row["strike"]), T, sigma, "call")
    if kind == "put":
        return bs_price(s0, float(row["strike"]), T, sigma, "put")
    if kind == "binary_put":
        return binary_put_price(s0, float(row["strike"]), T, sigma, float(row["payout"]))
    if kind == "chooser":
        # Simple chooser identity under r=0, same K for call/put at final expiry:
        # chooser = call(T_final, K) + put(T_choice, K)
        T_choice = float(row["choice_steps"]) / STEPS_PER_YEAR
        K = float(row["strike"])
        return bs_price(s0, K, T, sigma, "call") + bs_price(s0, K, T_choice, sigma, "put")
    if kind == "ko_put":
        return np.nan  # path-dependent discrete monitoring, MC only here
    return np.nan

priced["fair_analytic"] = priced.apply(lambda r: analytic_fair(r, S0, SIGMA), axis=1)
priced["mc_vs_analytic_abs_diff"] = np.abs(priced["fair_mc"] - priced["fair_analytic"])

cols = ["symbol","kind","fair_mc","se_mc","fair_analytic","mc_vs_analytic_abs_diff"]
print(priced[cols].sort_values("symbol").to_string(index=False))

In [ ]:
# ===============================
# Cell 8: Optimization (expected PnL) + conservative CI variant
# ===============================
def recommend_orders(df_priced, contract_size=3000, conservative=True):
    recs = []

    for _, r in df_priced.iterrows():
        fair = float(r["fair_mc"])
        bid = float(r["bid"])
        ask = float(r["ask"])
        max_buy = int(r["max_buy"])
        max_sell = int(r["max_sell"])

        # point estimate edges
        buy_edge = fair - ask
        sell_edge = bid - fair

        # conservative edges based on CI
        buy_edge_cons = float(r["ci_low"] - ask)
        sell_edge_cons = float(bid - r["ci_high"])

        if conservative:
            buy_ok = buy_edge_cons > 0
            sell_ok = sell_edge_cons > 0
        else:
            buy_ok = buy_edge > 0
            sell_ok = sell_edge > 0

        side = "NONE"
        qty = 0
        ev_unit = 0.0
        trigger_edge = 0.0

        # If both true (rare / only if extreme mispricing), choose larger EV edge
        if buy_ok and sell_ok:
            if buy_edge >= sell_edge:
                side, qty, ev_unit, trigger_edge = "BUY", max_buy, buy_edge, buy_edge_cons if conservative else buy_edge
            else:
                side, qty, ev_unit, trigger_edge = "SELL", max_sell, sell_edge, sell_edge_cons if conservative else sell_edge
        elif buy_ok:
            side, qty, ev_unit, trigger_edge = "BUY", max_buy, buy_edge, buy_edge_cons if conservative else buy_edge
        elif sell_ok:
            side, qty, ev_unit, trigger_edge = "SELL", max_sell, sell_edge, sell_edge_cons if conservative else sell_edge

        signed_qty = qty if side == "BUY" else (-qty if side == "SELL" else 0)
        ev_total = abs(signed_qty) * ev_unit * contract_size

        recs.append({
            "symbol": r["symbol"],
            "side": side,
            "volume": qty,
            "signed_qty": signed_qty,
            "fair_mc": fair,
            "bid": bid,
            "ask": ask,
            "buy_edge_point": buy_edge,
            "sell_edge_point": sell_edge,
            "trigger_edge_used": trigger_edge,
            "expected_pnl_per_unit": ev_unit,
            "expected_pnl_total_scaled": ev_total
        })

    rec_df = pd.DataFrame(recs).sort_values("symbol").reset_index(drop=True)
    total_ev = rec_df["expected_pnl_total_scaled"].sum()
    return rec_df, total_ev


orders_cons, total_ev_cons = recommend_orders(priced, contract_size=CONTRACT_SIZE, conservative=True)
orders_point, total_ev_point = recommend_orders(priced, contract_size=CONTRACT_SIZE, conservative=False)

print("Conservative-CI expected total PnL (scaled):", total_ev_cons)
display(orders_cons)

print("\nPoint-estimate expected total PnL (scaled):", total_ev_point)
display(orders_point)

In [ ]:
# ===============================
# Cell 9: Portfolio risk under 100-path scoring
# ===============================
def portfolio_single_path_stats(df_contracts, orders_df, s0, sigma, n_paths=1_000_000, chunk_size=200_000, seed=7, verbose=True):
    # map symbol -> order
    ord_map = {r["symbol"]: int(r["signed_qty"]) for _, r in orders_df.iterrows()}
    active = df_contracts[df_contracts["symbol"].isin([s for s, q in ord_map.items() if q != 0])].copy()
    if active.empty:
        return {"mean_single_scaled":0.0, "std_single_scaled":0.0, "mean_score100_scaled":0.0, "std_score100_scaled":0.0, "prob_score100_negative":0.0}

    max_steps = int(active["expiry_steps"].max())
    rng = np.random.default_rng(seed)

    sum_pnl = 0.0
    sumsq_pnl = 0.0
    done = 0

    while done < n_paths:
        m = min(chunk_size, n_paths - done)
        S = simulate_paths_chunk(m, max_steps, s0, sigma, rng)

        pnl = np.zeros(m, dtype=np.float64)
        for _, row in active.iterrows():
            sym = row["symbol"]
            q = ord_map[sym]
            if q == 0:
                continue

            pay = payoff_vector(row, S, s0)
            if q > 0:
                pnl += q * (pay - float(row["ask"]))
            else:
                pnl += (-q) * (float(row["bid"]) - pay)

        sum_pnl += np.sum(pnl)
        sumsq_pnl += np.dot(pnl, pnl)
        done += m

        if verbose and (done % (5 * chunk_size) == 0 or done == n_paths):
            print(f"Risk MC progress: {done:,} / {n_paths:,}")

    mean_single = sum_pnl / n_paths
    var_single = max(sumsq_pnl / n_paths - mean_single**2, 0.0)
    std_single = np.sqrt(var_single)

    # scale by contract size
    mean_single_scaled = mean_single * CONTRACT_SIZE
    std_single_scaled = std_single * CONTRACT_SIZE

    # score is average across 100 sims
    mean_score100 = mean_single_scaled
    std_score100 = std_single_scaled / np.sqrt(100.0)

    z = (0.0 - mean_score100) / std_score100 if std_score100 > 0 else np.inf
    prob_neg = norm.cdf(z) if np.isfinite(z) else (1.0 if mean_score100 < 0 else 0.0)

    return {
        "mean_single_scaled": mean_single_scaled,
        "std_single_scaled": std_single_scaled,
        "mean_score100_scaled": mean_score100,
        "std_score100_scaled": std_score100,
        "prob_score100_negative": prob_neg
    }

# Example risk report for conservative orders
risk_report = portfolio_single_path_stats(
    contracts, orders_cons, S0, SIGMA,
    n_paths=1_000_000, chunk_size=200_000, seed=11, verbose=True
)

risk_report

In [ ]:
# ===============================
# Cell 10: Final submission table format
# ===============================
def submission_table(orders_df):
    sub = orders_df.loc[orders_df["side"] != "NONE", ["symbol", "side", "volume"]].copy()
    sub = sub.sort_values("symbol").reset_index(drop=True)
    return sub

submission_cons = submission_table(orders_cons)
submission_point = submission_table(orders_point)

print("Conservative submission:")
display(submission_cons)

print("Point-estimate submission:")
display(submission_point)